[Guide Index](README.md)


# 02a Working with Datasets

`StarLayerDataset` wraps a single `pyoxigraph.Store` holding several independently addressable named graphs; each is an RDF-1.2-aware `StarLayerGraph` view of one graph of that store. `StarLayerDataset` is not an rdflib `Dataset`, and terms read from it are pyoxigraph terms.

By default a `StarLayerDataset` is a union: reading it (`triples`, `len`, `iter`, and the default graph of a SPARQL query) sees all graphs together, which is what `default_union=True` means. `default_union=False` gives rdflib's `Dataset` behaviour, where the default graph is only the default graph. `quads()` always shows graph names.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier sections.


In [1]:
import pyoxigraph as ox
from pyoxigraph import Literal, Triple

from starlayer.graph.oxgraph import StarLayerDataset, StarLayerGraph


class Ns:
    """A tiny namespace helper: Ns(base).name (or Ns(base)["name"]) is a pyoxigraph NamedNode."""
    def __init__(self, base): self.base = base
    def __getattr__(self, name): return ox.NamedNode(self.base + name)
    def __getitem__(self, name): return ox.NamedNode(self.base + name)


EX = Ns("http://example.org/")
RDF = Ns("http://www.w3.org/1999/02/22-rdf-syntax-ns#")

## 1. Creating and populating named graphs

`StarLayerDataset()` creates an empty dataset (in memory; `path=` opens an on-disk Oxigraph store).

`dataset.graph(<name>)` returns the graph for that name. If the graph does not exist it is created with the given name (in Oxigraph a named graph exists as soon as it holds a quad, or has been created with SPARQL `CREATE GRAPH`). The graph views share the dataset's prefix dictionary, so one `bind()` covers them all.

In [2]:
ds = StarLayerDataset()
ds.bind("ex", EX.base)

# create two empty StarLayerGraph views in the dataset
g1 = ds.graph(EX.hr)
g2 = ds.graph(EX.crm)

g1.add((EX.alice, EX.worksFor, EX.AcmeCorp))
g1.add((EX.alice, EX.title, Literal("Sales Manager")))
g2.add((EX.alice, EX.likes, EX.ProductABC))

print("g1 triple count:", len(g1))
print("g2 triple count:", len(g2))
print("graph names:", sorted(g.value for g in ds.graph_names()))

# TriG is the default format for dataset serialization (serialize() with no format is TriG)
print("\nResulting dataset")


print(ds.serialize(format="trig", pretty=True))

g1 triple count: 2
g2 triple count: 1
graph names: ['http://example.org/crm', 'http://example.org/hr']

Resulting dataset
@prefix ex: <http://example.org/> .

GRAPH ex:crm {
    ex:alice ex:likes ex:ProductABC .
}

GRAPH ex:hr {
    ex:alice ex:title "Sales Manager" ;
        ex:worksFor ex:AcmeCorp .
}



## 2. Attaching statements to a graph's own IRI, and reifying within a named graph

Each named graph has an identifying IRI.  That IRI is another resource node, so ordinary statements *about* the graph itself can be created.  Typically these statements would be added to the default graph or another named graph of the dataset, since a graph can best be described outside itself.

Reification works the same way.  The example below reifies a claim living inside `ex:hr`, and separately reifies a claim *about* `ex:hr` itself, living in the dataset's default graph.

In [3]:
ds = StarLayerDataset()
ds.bind("ex", EX.base)

g1 = ds.graph(EX.hr)
g1.add((EX.alice, EX.worksFor, EX.AcmeCorp))

# describe the ex:hr graph itself, in the dataset's default graph
ds.add((EX.hr, EX.maintainedBy, EX.DataTeam))

# reify a claim about a triple inside the ex:hr graph
# claim and reification exist in same graph
tt = Triple(EX.alice, EX.worksFor, EX.AcmeCorp)
g1.add_reification(EX.claim, tt)
g1.add((EX.claim, EX.source, EX.HRSystem))

# reification works the same way on a triple that lives in the dataset's own
# default graph - ds.default_graph is a StarLayerGraph view of it.
# reification lives in the default graph for a triple in the default graph
default_graph = ds.default_graph
graph_tt = Triple(EX.hr, EX.maintainedBy, EX.DataTeam)
default_graph.add_reification(EX.hrMeta, graph_tt)
default_graph.add((EX.hrMeta, EX.lastUpdated, Literal("2026-09-16")))

# serialize() of a dataset is TriG: ds.serialize(format="trig", pretty=True) equals ds.serialize())
print("\nResulting dataset")
print(ds.serialize(format="trig", pretty=True))

# note: the reifier and its annotation live inside GRAPH <ex:hr>, since that's the graph
# g1 refers to - while the statement about ex:hr itself, and its own reifier, live in
# the default graph.


Resulting dataset
@version "1.2" .
@prefix ex: <http://example.org/> .

ex:hr ex:maintainedBy ex:DataTeam ~ ex:hrMeta {| ex:lastUpdated "2026-09-16" |} .

GRAPH ex:hr {
    ex:alice ex:worksFor ex:AcmeCorp ~ ex:claim {| ex:source ex:HRSystem |} .
}



### 2.1 Keeping annotations in a separate graph from the facts they annotate

Nothing requires a reifier and its annotations to live in the same graph as the triple they describe — `Triple(s, p, o)` identifies a triple by value, not by which graph it's asserted in. This lets facts and their provenance/metadata be managed separately: one graph of raw facts, and a separate `annotation-graph` holding claims about those facts.

In [4]:
ds = StarLayerDataset()
ds.bind("ex", EX.base)

# the fact lives in ex:hr
g1 = ds.graph(EX.hr)
g1.add((EX.alice, EX.worksFor, EX.AcmeCorp))

# the annotation about that same fact lives in a completely separate graph
ann = ds.graph(EX["annotation-graph"])
tt = Triple(EX.alice, EX.worksFor, EX.AcmeCorp)
ann.add_reification(EX.claim, tt)
ann.add((EX.claim, EX.source, EX.HRSystem))

print("g1 triple count:", len(g1))
print("annotation-graph triple count:", len(ann))


print("\nResulting dataset")
print(ds.serialize(format="trig", pretty=True))

g1 triple count: 1
annotation-graph triple count: 2

Resulting dataset
@version "1.2" .
@prefix ex: <http://example.org/> .

GRAPH ex:annotation-graph {
    << ex:alice ex:worksFor ex:AcmeCorp ~ ex:claim >> ex:source ex:HRSystem .
}

GRAPH ex:hr {
    ex:alice ex:worksFor ex:AcmeCorp .
}



### 2.2 Using reifier functions with datasets

`StarLayerDataset` also has dataset-wide versions of `reifiers()`/`reifications()`/`reifier_annotations()`/
`reified_triples()`. Each returns a new `StarLayerDataset` including the matching triples, still scoped to the named graph that contains the reifier.

In [5]:
ds = StarLayerDataset()
ds.bind("ex", EX.base)

# the fact lives in ex:hr
g1 = ds.graph(EX.hr)
g1.add((EX.alice, EX.worksFor, EX.AcmeCorp))

# the annotation about that same fact lives in a separate graph
ann = ds.graph(EX["annotation-graph"])
tt = Triple(EX.alice, EX.worksFor, EX.AcmeCorp)
ann.add_reification(EX.claim, tt)
ann.add((EX.claim, EX.source, EX.HRSystem))

# ds.reifiers(TT=tt) searches all graphs in the dataset at once 
result = ds.reifiers(TT=tt)

print("triples found:", len(result))
print("\nResulting result dataset")
print(result.serialize(format="trig", pretty=True))

print("\nResulting full dataset")
print(ds.serialize(format="trig", pretty=True))

triples found: 2

Resulting result dataset
@version "1.2" .
@prefix ex: <http://example.org/> .

GRAPH ex:annotation-graph {
    << ex:alice ex:worksFor ex:AcmeCorp ~ ex:claim >> ex:source ex:HRSystem .
}


Resulting full dataset
@version "1.2" .
@prefix ex: <http://example.org/> .

GRAPH ex:annotation-graph {
    << ex:alice ex:worksFor ex:AcmeCorp ~ ex:claim >> ex:source ex:HRSystem .
}

GRAPH ex:hr {
    ex:alice ex:worksFor ex:AcmeCorp .
}



## 3. Serializing a dataset

A dataset can be serialized two ways, depending on whether graph boundaries need to be preserved:
- **3.1 TriG and N-Quads** — graph-preserving: `trig` (the default for `StarLayerDataset.serialize`) uses one `GRAPH` block per named graph, `nquads` is one quad per line (`s p o g`). Both are RDF 1.2 syntax on this class.
- **3.2 Plain Turtle** — every named graph flattened into a single set of triples; graph boundaries are lost.

### 3.1 TriG and N-Quads: graph-preserving serialization

Adding directly to the dataset (`ds.add(...)`, no graph specified) writes to the dataset's own default graph, separate from any named graph. Both `trig` and `nquads` preserve graph boundaries — see the [serialization formats guide](06a-serialization-formats.ipynb) for their single-graph counterparts and the rest of the format set. Parsing a document that has named graphs needs a `StarLayerDataset`; a `StarLayerGraph` view refuses a quad format whose document names graphs.

In [6]:
ds = StarLayerDataset()
ds.bind("ex", EX.base)

g1 = ds.graph(EX.hr)
g1.add((EX.alice, EX.worksFor, EX.AcmeCorp))

g2 = ds.graph(EX.crm)
g2.add((EX.alice, EX.likes, EX.ProductABC))

ds.add((EX.hr, EX.maintainedBy, EX.DataTeam))
print("\nResulting dataset in TriG")
print(ds.serialize(format="trig", pretty=True))
print("\nResulting dataset in N-Quads")
print(ds.serialize(format="nquads"))




Resulting dataset in TriG
@prefix ex: <http://example.org/> .

ex:hr ex:maintainedBy ex:DataTeam .

GRAPH ex:crm {
    ex:alice ex:likes ex:ProductABC .
}

GRAPH ex:hr {
    ex:alice ex:worksFor ex:AcmeCorp .
}


Resulting dataset in N-Quads
<http://example.org/hr> <http://example.org/maintainedBy> <http://example.org/DataTeam> .
<http://example.org/alice> <http://example.org/likes> <http://example.org/ProductABC> <http://example.org/crm> .
<http://example.org/alice> <http://example.org/worksFor> <http://example.org/AcmeCorp> <http://example.org/hr> .



### 3.2 Flattening a dataset into plain Turtle

Turtle has no syntax for multiple named graphs — plain Turtle is a single-graph format. `ds.serialize(format="turtle")` (also `nt`, `xml`, and `pretty=True`) handles this automatically: every graph is flattened into one set of triples, each distinct triple written once, and which graph each triple came from is lost. A dataset serialized to a triples-only format flattens its graphs; use `trig` or `nquads` to keep them.

`to_graph()` does the same flattening but returns the merged `StarLayerGraph` itself, for when the result needs to be queried or otherwise used as a graph rather than just printed.

In [7]:
ds = StarLayerDataset()
ds.bind("ex", EX.base)

g1 = ds.graph(EX.hr)
g1.add((EX.alice, EX.worksFor, EX.AcmeCorp))

g2 = ds.graph(EX.crm)
g2.add((EX.alice, EX.likes, EX.ProductABC))

ds.add((EX.hr, EX.maintainedBy, EX.DataTeam))

print(ds.serialize(format="turtle", pretty=True))

# to_graph() returns the merged StarLayerGraph itself, so it can be queried
# directly - not just printed
merged = ds.to_graph()
for row in merged.query("SELECT ?s WHERE { ?s ex:worksFor ?o }", initNs={"ex": EX.base}):
    print(row["s"].value)

@prefix ex: <http://example.org/> .

ex:alice ex:likes ex:ProductABC ;
    ex:worksFor ex:AcmeCorp .

ex:hr ex:maintainedBy ex:DataTeam .

http://example.org/alice
